# Session 4 · Part 2 — Too big for Excel

Excel stops at **1,048,576 rows**. Card networks, trading systems and banks produce that many records in minutes.
In this notebook you will:
1. **make** a file of 2 million transactions on your own laptop,
2. see what happens when Excel tries to open it,
3. load it in pandas, and learn three tricks for when data gets big: **pick columns, shrink types, read in chunks**,
4. see why you never loop over rows.

> If your laptop struggles (very slow, or a memory error), change `N_ROWS` below to `500_000` and run again.

In [ ]:
N_ROWS = 2_000_000

## 1 · Make the data
Run this cell. You don't need to understand the code. It creates a realistic but **fake** set of card transactions.

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd

out_dir = Path("data/generated")
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "transactions_big.csv"

start = time.perf_counter()
rng = np.random.default_rng(42)
cats = np.array(["Groceries", "Restaurants", "Transport", "Utilities", "Shopping",
                 "Travel", "Health", "Entertainment", "Education", "Rent"])
scale = np.array([1.5, 1.2, 0.6, 6, 3, 15, 3, 1.5, 20, 80])
cat_idx = rng.choice(len(cats), N_ROWS, p=[.2, .14, .12, .08, .14, .05, .06, .09, .04, .08])

big = pd.DataFrame({
    "transaction_id": np.arange(1, N_ROWS + 1),
    "timestamp": pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 365 * 24 * 3600, N_ROWS), unit="s"),
    "customer_id": rng.integers(10_000, 60_000, N_ROWS),
    "category": cats[cat_idx],
    "channel": rng.choice(["Card", "Online", "ATM", "Transfer"], N_ROWS, p=[.55, .3, .08, .07]),
    "country": rng.choice(["AE", "IN", "GB", "PK", "EG", "SA", "US"], N_ROWS, p=[.6, .12, .08, .05, .05, .05, .05]),
    "amount_aed": np.round(rng.lognormal(4.0, 0.8, N_ROWS) * scale[cat_idx], 2),
    "flagged": rng.random(N_ROWS) < 0.005,
})
big.to_csv(csv_path, index=False)
del big

print(f"Created {N_ROWS:,} rows in {time.perf_counter() - start:.1f} s")
print(f"File size: {csv_path.stat().st_size / 1e6:,.0f} MB  ->  {csv_path.resolve()}")

**Instructor demo:** open `data/generated/transactions_big.csv` in Excel. Watch for the *"File not loaded completely"* message: everything after row 1,048,576 is silently left out. Any total you calculate in that sheet would be wrong, without any warning in the cells themselves.

## 2 · Load it in pandas

In [ ]:
start = time.perf_counter()
big = pd.read_csv(csv_path)
print(f"Loaded {len(big):,} rows in {time.perf_counter() - start:.1f} s")
big.head()

In [ ]:
mb = big.memory_usage(deep=True) / 1e6
print(mb.round(1))
print(f"TOTAL: {mb.sum():,.0f} MB in memory")

All 2 million rows are there. Excel lost almost half of them. Now, three tricks for when data is big.

## 3 · Trick 1 — Only load the columns you need
Most questions need a few columns, not all of them.

In [ ]:
start = time.perf_counter()
slim = pd.read_csv(csv_path, usecols=["category", "amount_aed"])
print(f"{time.perf_counter() - start:.1f} s, {slim.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

## 4 · Trick 2 — Shrink the data types
`category` only has 10 different values, repeated 2 million times. The **category** type stores each word once plus a small code per row.

In [ ]:
before = big.memory_usage(deep=True).sum() / 1e6
for col in ["category", "channel", "country"]:
    big[col] = big[col].astype("category")
big["amount_aed"] = pd.to_numeric(big["amount_aed"], downcast="float")
after = big.memory_usage(deep=True).sum() / 1e6
print(f"Before: {before:,.0f} MB   After: {after:,.0f} MB   ({1 - after / before:.0%} smaller)")

⚠️ `downcast="float"` stores amounts with less precision (float32). That's fine for exploring and charts, **not** for the ledger. Same lesson as `0.1 + 0.2` in Part 1.

## 5 · Trick 3 — Read in chunks
What if the file were 200 million rows and didn't fit in memory at all? Read it **a piece at a time** and keep only the running totals.

In [ ]:
totals = None
for chunk in pd.read_csv(csv_path, usecols=["category", "amount_aed"], chunksize=500_000):
    part = chunk.groupby("category")["amount_aed"].sum()
    totals = part if totals is None else totals.add(part, fill_value=0)
    print("processed a chunk of", f"{len(chunk):,}", "rows")

totals.sort_values(ascending=False).round(0)

Same answer as loading everything, but memory use never went above one chunk.

## 6 · Never loop over rows
Say the bank charges 1% on every **Online** transaction. Here are two ways to calculate the fees.

In [ ]:
sample = big.head(100_000)

start = time.perf_counter()
fees = []
for _, row in sample.iterrows():                       # one row at a time, like a person with a calculator
    fees.append(row["amount_aed"] * 0.01 if row["channel"] == "Online" else 0.0)
t_loop = time.perf_counter() - start
print(f"Loop over {len(sample):,} rows: {t_loop:.1f} s  (so about {t_loop * len(big) / len(sample):.0f} s for all {len(big):,})")

start = time.perf_counter()
big["fee"] = np.where(big["channel"] == "Online", big["amount_aed"] * 0.01, 0.0)   # whole column at once
print(f"Vectorised, all {len(big):,} rows: {time.perf_counter() - start:.2f} s")

Whole-column operations (**vectorised**) are what make pandas fast. If you find yourself writing `for row in ...` over a DataFrame, there is almost always a one-line column version.

## 7 · The pivot table Excel couldn't do
**Race:** the instructor builds this pivot in Excel from `data/transactions_50k.xlsx` (50,000 rows, which Excel copes with). You build it here from **2 million**.

In [ ]:
pivot = big.pivot_table(index="category", columns="channel", values="amount_aed",
                        aggfunc="sum", observed=True).round(0)
pivot

## Tidy up
The generated files are large and are **excluded from Git** by the `.gitignore` file, so they won't be uploaded to GitHub. Delete them when you're done:

In [ ]:
for p in out_dir.glob("transactions_big.*"):
    p.unlink()
print("Deleted generated files")